# 00 · Foundations — Part 2: Reducers and Messages
### LangGraph · how updates combine

In Part 1, every node wrote to its *own* field, so nothing ever collided. Real graphs aren't that tidy. Two
checks may both want to say something about the same ticket, a loop may add to the same list on every pass,
and a chat keeps growing a list of messages.

So LangGraph needs a rule for **what happens when a field is written more than once**. That rule is called a
**reducer**.

Think of each box on the whiteboard as having a small label saying how new writing is handled:

| Label on the box | Meaning | In code |
|---|---|---|
| *(no label)* | **Overwrite** — the newest value replaces the old one | a plain type, such as `str` |
| *"Add to the pile"* | **Append** — new items are added to the old ones | `Annotated[list, operator.add]` |
| *"Keep the larger"* | **Combine with a rule you write** | `Annotated[str, my_function]` |

A reducer is just a function `f(old, new) → combined`. That's all it is.

> 📌 **Checked while building this notebook** (`langgraph 1.2.17`; every point below was observed by running it)
> - **Without a reducer, the last write wins.** Sequential nodes overwrite each other.
> - **Two parallel nodes writing the same field without a reducer fail loudly** with `InvalidUpdateError`
>   ("Can receive only one value per step. Use an Annotated key to handle multiple values").
> - **A field with a reducer can be left out of the input.** It starts out as the *empty value of its type*
>   (`''`, `0`, `[]`, `{}`), and your reducer is run on that empty value too, so a custom reducer must cope
>   with it. (A first draft of this notebook didn't, and crashed with `KeyError: ''`.)
> - **Parallel appends came out in the order the nodes were added** (identical across five runs). That's
>   observed behaviour, not a promise, so don't build logic that depends on it.
> - **`add_messages` is smarter than "append"**: it accepts strings, tuples and dicts, replaces a message that
>   has the same `id`, and deletes with `RemoveMessage`.
> - **Trimming a chat by count can leave the history starting with an AI reply.** Section 5 shows a safe way.
> - `MessageGraph` is deprecated since LangGraph 1.0; use `StateGraph` with a `messages` key (`MessagesState`).
> - Sections 1–5 run offline with no API key. The capstone has a built-in stand-in model so it can too.

## Setup

In [ ]:
# %pip install -U langgraph langchain python-dotenv
# %pip install -U langchain-openai     # only if you use LLM=openai:...   (the default)
# %pip install -U langchain-groq       # only if you use LLM=groq:...


In [ ]:
import operator
from typing import TypedDict, Annotated

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage, trim_messages
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages

print("Ready.")


---
## 1. The default: the last write wins

Here is a ticket whose priority is set by two nodes in a row. First a rule-based check sets it, then a VIP
check upgrades it. With no reducer on `priority`, each write simply replaces the previous one.

In [ ]:
class Ticket(TypedDict):
    amount: float
    is_vip: bool
    priority: str


def rule_check(state: Ticket) -> dict:
    return {"priority": "high" if state["amount"] > 1000 else "medium"}

def vip_check(state: Ticket) -> dict:
    return {"priority": "high"} if state["is_vip"] else {}


def build(first, second):
    b = StateGraph(Ticket)
    b.add_node("first", first); b.add_node("second", second)
    b.add_edge(START, "first"); b.add_edge("first", "second"); b.add_edge("second", END)
    return b.compile()

start = {"amount": 50, "is_vip": True}
print("rule_check, then vip_check :", build(rule_check, vip_check).invoke(start)["priority"])
print("vip_check, then rule_check :", build(vip_check, rule_check).invoke(start)["priority"], " <- order changed the answer")


The result depends on the **order**, because the second write throws away the first. That's fine when you want
it ("the latest value is the truth"), but it's a trap when you wanted *both* results kept.

---
## 2. Reducers: keeping everything

Put `Annotated[<type>, <reducer>]` on a field and LangGraph will *combine* writes with that function instead of
overwriting. The standard library already has the common ones: `operator.add` joins lists, adds numbers, and
concatenates strings.

A reducer is only a function of `(old, new)`, so you can try one by hand:

In [ ]:
print(operator.add(["billing check passed"], ["fraud check passed"]))
print(operator.add(2, 1))


Now a ticket pipeline that collects an audit trail of everything that happened to it, and counts the
attempts:

In [ ]:
class Audited(TypedDict):
    notes: Annotated[list, operator.add]       # every write is appended
    attempts: Annotated[int, operator.add]     # every write is added to the total


def receive(state: Audited) -> dict:
    return {"notes": ["received"], "attempts": 1}

def validate(state: Audited) -> dict:
    return {"notes": ["validated"], "attempts": 1}

def assign(state: Audited) -> dict:
    return {"notes": ["assigned to billing"], "attempts": 1}


b = StateGraph(Audited)
for name, fn in [("receive", receive), ("validate", validate), ("assign", assign)]:
    b.add_node(name, fn)
b.add_edge(START, "receive"); b.add_edge("receive", "validate"); b.add_edge("validate", "assign"); b.add_edge("assign", END)
audited = b.compile()

# Note: we pass NO value for notes or attempts. A field with a reducer starts out empty.
print(audited.invoke({}))


Every node returned a *one-item* list (`["received"]`), and the reducer stitched them into one. You don't
need to read the old list and add to it yourself. That is the point.

---
## 3. Parallel nodes writing the same field

Reducers matter most when nodes run at the same time. Imagine two checks that each judge a ticket's priority
in the same super-step.

In [ ]:
class Judged(TypedDict):
    amount: float
    text: str
    priority: str                     # NO reducer


def check_amount(state: Judged) -> dict:
    return {"priority": "high" if state["amount"] > 1000 else "low"}

def check_keywords(state: Judged) -> dict:
    return {"priority": "medium" if "refund" in state["text"].lower() else "low"}


def build_parallel(schema):
    b = StateGraph(schema)
    b.add_node("check_amount", check_amount); b.add_node("check_keywords", check_keywords)
    b.add_edge(START, "check_amount"); b.add_edge(START, "check_keywords")      # both start together
    b.add_edge("check_amount", END); b.add_edge("check_keywords", END)
    return b.compile()


try:
    build_parallel(Judged).invoke({"amount": 2500, "text": "I want a refund"})
except Exception as error:
    print(type(error).__name__, "->", str(error).splitlines()[0])


LangGraph refuses to guess which of two simultaneous values should win, and it says how to fix it: use an
`Annotated` key. For priorities the sensible rule is **"keep the highest"**, so we write a custom reducer:

In [ ]:
RANK = {"low": 0, "medium": 1, "high": 2}

def highest_priority(old: str, new: str) -> str:
    """Reducer: of two priorities, keep the more urgent one. (`old` is '' the very first time: see below.)"""
    return old if RANK.get(old, -1) >= RANK.get(new, -1) else new


class JudgedFixed(TypedDict):
    amount: float
    text: str
    priority: Annotated[str, highest_priority]


fixed = build_parallel(JudgedFixed)
print(fixed.invoke({"amount": 2500, "text": "I want a refund"})["priority"], " <- amount says high, keywords say medium")
print(fixed.invoke({"amount": 20,   "text": "I want a refund"})["priority"], " <- amount says low,  keywords say medium")


Look closely at `RANK.get(old, -1)`. A first draft of this cell used `RANK[old]` and crashed with
`KeyError: ''`. The reason is worth understanding, because it will happen to you.

**Before any node writes, a field holds the *empty value of its type*** (`''` for text, `0` for a number, `[]`
for a list, `{}` for a dict). LangGraph runs your reducer on that empty value too. Let's watch it happen:

In [ ]:
calls = []

def watching(old, new):
    calls.append((old, new))
    return new


class Watched(TypedDict):
    note: Annotated[str, watching]


def write_note(state: Watched) -> dict:
    return {"note": "written by the node"}

b = StateGraph(Watched); b.add_node("write", write_note); b.add_edge(START, "write"); b.add_edge("write", END)
b.compile().invoke({"note": "the starting value"})

for old, new in calls:
    print(f"reducer(old={old!r}, new={new!r})")


The very first call is `reducer('', 'the starting value')`: your input is itself a "write" that goes through
the reducer, and the old value it sees is the empty string. **Rule: a custom reducer must cope with the type's
empty value.** (`operator.add` does, naturally, since `[] + [...]` and `0 + 3` just work.)

Two more reducers cover most real needs. **Collecting flags** from several parallel checks into one list, and
**merging dictionaries** with `operator.or_`:

In [ ]:
class Screened(TypedDict):
    text: str
    flags: Annotated[list, operator.add]          # each check may add flags
    fields: Annotated[dict, operator.or_]         # each check may add extracted fields (dicts are merged)


def spam_check(state: Screened) -> dict:
    return {"flags": ["spam"]} if "free money" in state["text"].lower() else {}

def pii_check(state: Screened) -> dict:
    found = "card" in state["text"].lower()
    return {"flags": ["contains payment info"], "fields": {"pii": True}} if found else {"fields": {"pii": False}}

def language_check(state: Screened) -> dict:
    return {"fields": {"language": "en"}}


b = StateGraph(Screened)
for name, fn in [("spam", spam_check), ("pii", pii_check), ("language", language_check)]:
    b.add_node(name, fn); b.add_edge(START, name); b.add_edge(name, END)

result = b.compile().invoke({"text": "FREE MONEY! my card number is on file"})
print("flags :", sorted(result["flags"]))        # sorted: the order of parallel appends isn't a promise
print("fields:", result["fields"])


Three checks ran together, each contributed what it knew, and nothing was overwritten or lost.

---
## 4. Messages: a reducer built for conversations

A chat is a list of messages, and appending isn't quite enough. You also want to **edit** a message, **delete**
one, and accept messages written casually. LangGraph ships a reducer for exactly this, `add_messages`. Because it
is a plain function you can try it directly:

In [ ]:
# 1. It accepts casual formats and turns them into real message objects
combined = add_messages([], ["hello", ("user", "a tuple works too"), {"role": "assistant", "content": "so does a dict"}])
print([f"{type(m).__name__}: {m.content}" for m in combined])


In [ ]:
# 2. Messages get an id automatically; a new message with the SAME id replaces the old one
history = add_messages([], [HumanMessage("my order is late", id="m1"), AIMessage("sorry to hear that", id="m2")])
edited = add_messages(history, [HumanMessage("my order is VERY late", id="m1")])
print([(m.id, m.content) for m in edited])


In [ ]:
# 3. RemoveMessage deletes by id
trimmed = add_messages(edited, [RemoveMessage(id="m2")])
print([(m.id, m.content) for m in trimmed])


Because chats are so common, LangGraph provides `MessagesState`, a ready-made state with one field,
`messages`, already wired to `add_messages`. Extend it to add your own fields:

In [ ]:
class SupportChat(MessagesState):
    ticket_id: str          # our own field, next to the built-in `messages`

print("fields:", list(SupportChat.__annotations__))


---
## 5. Keeping a long conversation under control

Every message in the history is sent to the model on **every** turn. That costs money and, eventually, hits the
model's limit. So long chats need trimming. There are two different ideas, and it matters which you choose:

| | Removes messages from the **saved state**? | What the model sees |
|---|---|---|
| **A. `RemoveMessage` in a node** | **Yes**, permanently | The shortened history |
| **B. `trim_messages(...)` when calling the model** | **No**, everything stays in state | A shortened *view* |

**A** saves storage and is what we'll use below. **B** is better when you want a full transcript kept but a short
prompt sent. Here is B on a small example (counting messages with `len` keeps it simple; in practice you would
count tokens):

In [ ]:
conversation = [SystemMessage("You are a support assistant."), HumanMessage("1"), AIMessage("a1"),
                HumanMessage("2"), AIMessage("a2"), HumanMessage("3")]

view = trim_messages(conversation, max_tokens=3, token_counter=len, strategy="last", include_system=True, start_on="human")
print("what the model would see:", [m.content for m in view])
print("what is still in state   :", [m.content for m in conversation])


Now A, with one safety rule learned the hard way. If you just keep "the last N messages", the kept history can
begin with an **AI reply** (its question was cut off). Many models handle that badly. So after cutting by count we
also drop any leading non-human messages:

In [ ]:
def messages_to_remove(messages: list, keep_last: int) -> list:
    """Which messages to delete so at most `keep_last` remain, and the history starts on a human message."""
    kept = messages[-keep_last:]
    while kept and kept[0].type != "human":
        kept = kept[1:]
    dropped = messages[: len(messages) - len(kept)]
    return [RemoveMessage(id=m.id) for m in dropped]


demo = add_messages([], [HumanMessage("q1", id="1"), AIMessage("a1", id="2"), HumanMessage("q2", id="3"),
                         AIMessage("a2", id="4"), HumanMessage("q3", id="5"), AIMessage("a3", id="6"), HumanMessage("q4", id="7")])
removals = messages_to_remove(demo, keep_last=4)
after = add_messages(demo, removals)
print("removed ids :", [r.id for r in removals])
print("remaining   :", [m.content for m in after], " <- starts on a human message, not 'a2'")


---
## Real-world capstone: a support chatbot with bounded memory

A graph with two nodes: **`trim`** keeps the history short, then **`assistant`** answers. This is the pattern
behind almost every production chatbot.

```
 START → trim → assistant → END
```

### Choosing a model

One setting picks the provider, written as `provider:model`. Model names change often, so keep it in one place.

| Want | Set `LLM` to | Needs |
|---|---|---|
| OpenAI (default) | `openai:gpt-4o-mini` | `OPENAI_API_KEY` |
| Groq, free tier | `groq:openai/gpt-oss-120b` | `GROQ_API_KEY` |
| No key at all | `fake` | nothing: a built-in stand-in replies with fixed text |

Put it in your `.env` (for example `LLM=groq:openai/gpt-oss-120b`), or set it in the cell below. If you get a
`404 model_not_found`, the model name has been retired; pick a current one from your provider.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.language_models import FakeListChatModel

LLM = os.getenv("LLM", "openai:gpt-4o-mini")        # change here, or set LLM in your .env

if LLM == "fake":
    llm = FakeListChatModel(responses=["(stand-in reply: no model is connected)"])
else:
    llm = init_chat_model(LLM, temperature=0)

print("Using:", LLM)


In [ ]:
class ChatState(MessagesState):
    pass

MAX_KEPT = 6                      # keep at most 6 earlier messages (about 3 exchanges)

SYSTEM = SystemMessage("You are a concise, friendly support assistant for an online store.")


def trim(state: ChatState) -> dict:
    return {"messages": messages_to_remove(state["messages"], MAX_KEPT)}


def assistant(state: ChatState) -> dict:
    reply = llm.invoke([SYSTEM] + state["messages"])      # the system prompt is added per call, never stored,
    return {"messages": [reply]}                           # so trimming can never delete it


builder = StateGraph(ChatState)
builder.add_node("trim", trim)
builder.add_node("assistant", assistant)
builder.add_edge(START, "trim")
builder.add_edge("trim", "assistant")
builder.add_edge("assistant", END)

chatbot = builder.compile()


Run a conversation. After every turn we hand the returned messages back in, plus the new question. (Because
`add_messages` matches by `id`, the old messages are recognised and not duplicated. In `07_Memory_Persistence` a
*checkpointer* will do this remembering for you.)

In [ ]:
questions = ["Where is my order?", "It was supposed to arrive Monday.", "Can I change the address?",
             "What about a refund?", "How long does a refund take?", "Thanks, that's all!"]

state = {"messages": []}
sizes = []
for turn, question in enumerate(questions, start=1):
    state = chatbot.invoke({"messages": state["messages"] + [("user", question)]})
    sizes.append(len(state["messages"]))
    roles = "".join(m.type[0].upper() for m in state["messages"])          # H = human, A = ai
    print(f"turn {turn}: {len(state['messages'])} messages kept   {roles}")

assert max(sizes) <= MAX_KEPT + 1, sizes                              # never more than the cap plus the new reply
assert state["messages"][0].type == "human"                           # history always starts on a human message
print("\nMemory stayed bounded, and always began on a human message.")


The history grows for the first few turns, then **levels off**, no matter how long the conversation runs.
Each turn costs about the same, and the oldest context quietly drops away.

### Try it yourself
1. Change `MAX_KEPT` to 2, then to 20. How does the pattern of `H` and `A` change after the cap is reached?
2. Remove the `while` loop from `messages_to_remove` and run the conversation. Can you find a turn where the
   kept history begins with `A`?
3. Replace the `RemoveMessage` approach with `trim_messages` (idea B from Section 5) inside `assistant`. What
   is different about `state["messages"]` afterwards?
4. Add a `ticket_id` field (use `SupportChat` from Section 4) and make `assistant` mention it in its system
   prompt.
5. Add a node `summarise` that, before trimming, writes a one-line summary of the messages about to be
   removed into a second state field. (You'll need that field's reducer to be an overwrite.)

---
**Next:** `03_schemas_validation_and_context.ipynb` — choosing what *kind* of object your state is
(`TypedDict`, dataclass or Pydantic), what each one actually validates, who gets to see which fields, and how to
pass settings that don't belong in state.
